# 📱 AetherControl — Notebook del Desarrollador & Estudiante
**Proyecto:** AetherNet IoT & Autonomous Rover — App `AetherControl` (Kotlin + Jetpack Compose)  
**Stack:** Kotlin, MVVM, StateFlow, Coroutines, Retrofit, DataStore, Navigation Compose  
**Ubicación real del código:** `app/src/main/java/com/aethernet/aethercontrol/`  
**PRD / Requisitos:** `docs/prd.md`, `docs/requirements.md` (HU-01, HU-02, HU-03)  
**Autor:** Notebook didáctico para quien **recién empieza** en Kotlin/Android — con notas de dev, decisiones de arquitectura, sintaxis y ejercicios.

> **Cómo usar este notebook:**
> 1. Lee los bloques grises (markdown) — son la teoría con analogías simples.
> 2. Ejecuta las celdas de código (Shift+Enter) — son Python que **simula** Kotlin para que veas el resultado sin instalar Android Studio.
> 3. Copia los bloques ` ```kotlin ` a Android Studio cuando quieras probar en el proyecto real.
> 4. Haz los **Ejercicios ✏️** — tienen solución oculta más abajo.


---
## 📑 Índice
1. [Qué hicimos en Android Studio — Mapa en 5 minutos](#mapa)
2. [Notas de Desarrollador / Estudiante — Cómo pensar el código](#notas-dev)
3. [Decisiones de Arquitectura — Por qué lo hicimos así (ADR)](#adr)
4. [Kotlin — Palabras reservadas](#reservadas)
5. [Kotlin — Sintaxis desde cero](#sintaxis)
   - 5.1 `val` vs `var` — Caja sellada vs caja abierta
   - 5.2 Tipos básicos
   - 5.3 Null Safety (`?`, `?:`, `!!`)
   - 5.4 Listas y Colecciones
   - 5.5 Condicionales (`if`, `when`)
   - 5.6 Bucles (`for`, `while`, `repeat`)
   - 5.7 Funciones (`fun`)
   - 5.8 Clases, `data class`, `object`, `sealed interface`
   - 5.9 Lambdas
6. [Jetpack Compose — Legos que se redibujan solos](#compose)
7. [MVVM en AetherControl — del código real al diagrama](#mvvm)
8. [Ejercicios Integradores — Mini AetherControl en Python/Kotlin](#ejercicios)
9. [Checklist y Siguientes pasos](#checklist)


<a id="mapa"></a>
---
## 1. 🗺️ Qué hicimos en Android Studio — Mapa en 5 minutos

Si abres `app/src/main/java/com/aethernet/aethercontrol/` verás esto:
```
com.aethernet.aethercontrol/
├── MainActivity.kt              ← PUERTA de entrada (onCreate)
├── AetherControlApp.kt          ← DESPERTADOR (inicia ServiceLocator)
├── core/di/ServiceLocator.kt    ← CAJA DE HERRAMIENTAS (singleton manual)
├── data/
│   ├── remote/ApiService.kt     ← TELÉFONO al backend (Retrofit)
│   ├── remote/dto/*.kt          ← SOBRES (qué forma tienen los datos JSON)
│   ├── repository/AetherRepository.kt + Impl.kt ← COCINA (lógica de datos)
│   └── local/PreferencesManager.kt + AppDatabase.kt ← CUADERNITO (DataStore/Room)
├── domain/model/UiModels.kt     ← TICKET (qué ve la pantalla)
├── ui/
│   ├── screens/DashboardScreen.kt ← MESA (lo que ve el usuario)
│   ├── viewmodel/DashboardViewModel.kt ← MESERO (cerebro)
│   ├── navigation/NavGraph.kt   ← GPS
│   └── theme/                   ← PINTURA (colores/tipografía)
└── util/Result.kt               ← SEMÁFORO (Success / Error / Loading)
```

**Analogía del restaurante (MVVM):**
| Capa | Rol | Archivo |
|---|---|---|
| **View** (Mesa) | Solo dibuja y escucha. No hace cuentas. | `DashboardScreen.kt:35` |
| **ViewModel** (Mesero) | Pide a cocina, guarda estado en `StateFlow`, avisa a la mesa. | `DashboardViewModel.kt:18` |
| **Repository** (Cocina) | Habla con internet/BD, traduce errores a `Result`. | `AetherRepositoryImpl.kt:16` |
| **ApiService** (Teléfono) | Traduce `fun getHealth()` → `GET /health`. | `ApiService.kt:22` |

**Flujo de un click "Reintentar":** `Botón` → `vm.refreshHealth()` → `repo.getHealth()` → `api.getHealth()` → `Backend FastAPI /health` → `Result.Success` → `StateFlow` → `Compose se redibuja`


In [ ]:
# Celda 1 — Explora la estructura real (ejecútala)
from pathlib import Path
base = Path("app/src/main/java/com/aethernet/aethercontrol")
if not base.exists():
    base = Path("../app/src/main/java/com/aethernet/aethercontrol")
if not base.exists():
    base = Path("/home/craos6518/Documentos/AetherNet-IoT-Autonomous-Rover/app/src/main/java/com/aethernet/aethercontrol")
print(f"Base: {base} existe={base.exists()}")
for p in sorted(base.rglob("*.kt")):
    print(p.relative_to(base), f"— {p.stat().st_size} bytes")


<a id="notas-dev"></a>
---
## 2. 📝 Notas de Desarrollador / Estudiante — Cómo pensar el código

### Para el estudiante (recién empieza):
- **No leas de arriba a abajo.** Lee en orden: `UiModels.kt` → `Result.kt` → `ApiService.kt` → `AetherRepository.kt` → `DashboardViewModel.kt` → `DashboardScreen.kt`. Del dato más simple al más complejo.
- **Kotlin es 90% como Python pero con tipos.** Si sabes `x = 5` en Python, en Kotlin es `val x: Int = 5`. Lo extra es el tipo y si puede ser nulo.
- **Compose es declarativo:** No dices "cambia el texto a X", dices "el texto ES X si el estado es Y". El framework redibuja solo.

### Para el desarrollador (cuando toques AetherControl):
- **Observación 1 — Cuándo usar `val` vs `var`:** Siempre `val` (inmutable) por defecto. Solo `var` si Compose necesita `mutableStateOf` o el ViewModel necesita `_uiState`. En `DashboardScreen.kt:43` `var urlInput by remember` es `var` porque el usuario escribe; `val state by vm.uiState...` es `val` porque solo escuchas.
- **Observación 2 — Cuándo usar `StateFlow` vs `LiveData`:** En este proyecto **solo StateFlow** (`DashboardViewModel.kt:22`). LiveData es legacy. StateFlow + `collectAsStateWithLifecycle()` respeta ciclo de vida.
- **Observación 3 — Cuándo usar `remember` vs `rememberSaveable`:** `remember` sobrevive a recomposiciones, pero no a rotar pantalla. Si quieres que `urlInput` sobreviva a rotación, cambia a `rememberSaveable`. Hoy usamos `remember(savedUrl)` porque se resetea si cambia la URL guardada.
- **Observación 4 — Cuándo usar `suspend`:** Toda función que hace red/BD/archivo debe ser `suspend`. `ApiService.kt:28` `suspend fun getHealth()` y `AetherRepository.kt:21` `suspend fun getHealth()` — se llaman solo dentro de `viewModelScope.launch`.
- **Observación 5 — Cuándo usar `ServiceLocator`:** Solo porque `docs/requirements.md` RNF-3.1 exige FOSS sin Hilt/Koin. Si el proyecto creciera a 10 ViewModels, migrar a Hilt sería ADR futuro.
- **Tip de debug:** `ServiceLocator.getCurrentBaseUrl()` te dice qué URL está usando Retrofit realmente. El `Tip` en `DashboardScreen.kt:132` recuerda: emulador=`10.0.2.2:8000`, físico=`IP del PC`.


<a id="adr"></a>
---
## 3. 🏛️ Decisiones de Arquitectura (ADR) — Por qué lo hicimos así

| ADR | Decisión | Alternativa descartada | Por qué | Archivo |
|---|---|---|---|---|
| **ADR-01** | **MVVM + Repository** | MVC / Sin capas | `requirements.md` exige tests BDD; MVVM separa UI de lógica y permite testear ViewModel sin Android. | `DashboardViewModel.kt:18`, `AetherRepository.kt:19` |
| **ADR-02** | **ServiceLocator manual (object singleton)** | Hilt / Koin | RNF-3.1 FOSS 100% y simplicidad Sprint 1. Hilt añade complejidad de setup; Koin es FOSS pero se consideró extra para 1 ViewModel. Reevaluar en Sprint 3 si hay >3 VMs. | `ServiceLocator.kt:30` |
| **ADR-03** | **Retrofit + OkHttp + kotlinx.serialization** | Ktor / Volley | FOSS, estándar de facto, `asConverterFactory` con `Json { ignoreUnknownKeys=true }` tolera cambios del backend sin romper app. | `ServiceLocator.kt:87`, `ApiService.kt:22` |
| **ADR-04** | **DataStore Preferences (no Room aún, no SharedPreferences)** | SharedPreferences / Room | DataStore es async (`Flow`), lifecycle-aware y sin bloqueos. Room está preparado (`AppDatabase.kt`) pero sin entidades aún (Sprint 1 solo config). | `PreferencesManager.kt:20` |
| **ADR-05** | **StateFlow + `collectAsStateWithLifecycle` (no LiveData)** | LiveData | StateFlow es Kotlin puro, testeable con Turbine (`build.gradle.kts:71` `turbine`), y `ViewModel.viewModelScope` lo gestiona. | `DashboardViewModel.kt:22`, `DashboardScreen.kt:37` |
| **ADR-06** | **Result sealed interface + safeCall** | Try/catch en ViewModel /Throw | Centraliza manejo `IOException`/`HttpException` en `util/Result.kt:20`, ViewModel solo hace `when (repo.getHealth())`. | `Result.kt:10` |
| **ADR-07** | **Navigation Compose con `sealed class Dest`** | Fragmentos XML | Compose-first, menos boilerplate; `NavGraph.kt:14` listo para agregar `Dest.Rover`, `Dest.Settings`. | `NavGraph.kt:14` |

> **Regla de oro (`AGENTS.md:4`):** Antes de cambiar protocolo (RF/UART/MQTT) o umbrales (PIN, EMA α), pide confirmación humana. `ServiceLocator.normalizeUrl` es un umbral "blando" pero afecta conectividad.


<a id="reservadas"></a>
---
## 4. 🔑 Kotlin — Palabras reservadas (las que NO puedes usar como nombre de variable)

Kotlin tiene ~40 palabras que el compilador usa. Si intentas `val fun = 5` fallará. Las más importantes para AetherControl:


| Palabra | Para qué se usa | Ejemplo en AetherControl |
|---|---|---|
| `package`, `import` | Organizar archivos | `package com.aethernet...` `MainActivity.kt:1` |
| `val`, `var` | Declarar variable (inmutable/mutable) | `val state by vm.uiState...` `DashboardScreen.kt:37`, `var urlInput` `DashboardScreen.kt:43` |
| `fun` | Función | `fun refreshHealth()` `DashboardViewModel.kt:29` |
| `class`, `object`, `interface`, `data`, `sealed` | Crear tipos | `class DashboardViewModel`, `object ServiceLocator`, `interface ApiService`, `data class DashboardUiState`, `sealed interface Result` |
| `if`, `else`, `when` | Condicionales | `if (!state.isConnected)` `DashboardScreen.kt:55`, `when (val r = repo.getHealth())` `DashboardViewModel.kt:32` |
| `for`, `while`, `return`, `break`, `continue` | Bucles/control | `for (v in noisy)` en tests EMA |
| `null`, `true`, `false` | Valores literales | `val health: HealthResponse? = null` `UiModels.kt:11` |
| `suspend` | Función que puede pausarse (coroutine) | `suspend fun getHealth()` `AetherRepository.kt:21` |
| `override`, `private`, `public` | Modificadores | `override fun onCreate` `MainActivity.kt:14`, `private val _uiState` `DashboardViewModel.kt:22` |
| `by`, `in`, `is`, `as` | Delegación / chequeo tipos | `by viewModel()`, `is Result.Success`, `asStateFlow()` |
| `try`, `catch`, `throw` | Errores | `try { Result.Success(block()) } catch (e: IOException)` `Result.kt:20` |
| `this`, `super` | Referencia a sí mismo / padre | `super.onCreate(savedInstanceState)` `MainActivity.kt:15` |

**Truco:** En Android Studio las palabras reservadas se pintan de **naranja/morado**. Si ves tu nombre de variable en ese color, ¡cámbialo!

Lista completa oficial (no necesitas memorizar, solo reconocer): `as`, `break`, `class`, `continue`, `do`, `else`, `false`, `for`, `fun`, `if`, `in`, `interface`, `is`, `null`, `object`, `package`, `return`, `super`, `this`, `throw`, `true`, `try`, `typealias`, `val`, `var`, `when`, `while`, `by`, `catch`, `constructor`, `delegate`, `dynamic`, `field`, `file`, `finally`, `get`, `import`, `init`, `param`, `property`, `receiver`, `set`, `setparam`, `where`, `actual`, `abstract`, `annotation`, `companion`, `const`, `crossinline`, `data`, `enum`, `expect`, `external`, `final`, `internal`, `lateinit`, `noinline`, `open`, `operator`, `out`, `override`, `private`, `protected`, `public`, `reified`, `sealed`, `suspend`, `tailrec`, `vararg`.


<a id="sintaxis"></a>
---
## 5. 🧩 Kotlin — Sintaxis desde cero (con ejemplos de AetherControl)

En cada sección: **1) Qué es en simple → 2) Sintaxis Kotlin → 3) Equivalente Python → 4) Cuándo usarlo en AetherControl.**


### 5.1 `val` vs `var` — Caja sellada vs caja abierta

- `val` = **value**, no cambia nunca (como `const` o tu número de cédula). Preferir siempre.
- `var` = **variable**, sí cambia (como tu saldo bancario).

```kotlin
// DashboardScreen.kt:37  val = solo lectura, escucha al ViewModel
val state by vm.uiState.collectAsStateWithLifecycle()
// DashboardScreen.kt:43  var = el usuario puede escribir y borrar
var urlInput by remember(savedUrl) { mutableStateOf(savedUrl) }
urlInput = "http://192.168.1.50:8000/" // ✅ permitido porque es var
// state = ... // ❌ ERROR: val cannot be reassigned
```

**Regla:** Empieza con `val`. Solo cambia a `var` si el compilador te obliga o es estado de UI editable.


In [ ]:
# 🐍 Python simula val/var (Kotlin real en el bloque de arriba)
# En Python no hay val real, pero lo simulamos:
from dataclasses import dataclass

# val = tupla / frozen dataclass (no cambia)
VAL_PI = 3.1416
# VAL_PI = 3 # en Kotlin esto sería ERROR

# var = variable normal
url_input = "http://10.0.2.2:8000/"
print(f"Inicial: {url_input}")
url_input = "http://192.168.1.50:8000/" # var permite reasignar
print(f"Cambiada (var): {url_input}")

# Demo val vs var en lista
val_list = [1,2,3] # en Kotlin: val list = listOf(1,2,3) -> la referencia no cambia, pero si es mutableListOf sí cambia contenido
var_counter = 0
var_counter += 1
print(f"counter var: {var_counter}")

# Ejercicio: ¿Qué pasa si intentas cambiar un val?
# En Kotlin: val x = 5; x = 6 -> Compile error: Val cannot be reassigned
print("✅ val = no reasignas la variable. var = sí puedes.")


### 5.2 Tipos básicos — Como en Python pero con apellido obligatorio

| Kotlin | Python equiv. | Ejemplo | En AetherControl |
|---|---|---|---|
| `Int` | `int` | `val port: Int = 8000` | `port` en URL |
| `Double` | `float` | `val temp: Double = 36.6` | Telemetría rover |
| `String` | `str` | `val host: String = "10.0.2.2"` | `PreferencesManager.kt:28` |
| `Boolean` | `bool` | `val isConnected: Boolean = false` | `UiModels.kt:10` |
| `Long` | `int` | `val lastSync: Long? = System.currentTimeMillis()` | `UiModels.kt:13` (milis) |
| `Float` | `float` | `val alpha: Float = 0.2f` | EMA α |

```kotlin
var nombre: String = "Aether"
var edad: Int = 5
var conectado: Boolean = true
// Kotlin infiere el tipo si no lo pones:
var ciudad = "Manizales" // String inferido
```

**Tip:** En Kotlin `Int` y `Double` son objetos, no primitivos. `System.currentTimeMillis()` devuelve `Long` (por eso `lastSync: Long?`).


In [ ]:
# Tipos en Python (dinámico) vs Kotlin (estático)
nombre: str = "Aether"
edad: int = 5
conectado: bool = True
last_sync: int | None = 1714523456789 # Long? en Kotlin

print(f"nombre={nombre} ({type(nombre).__name__}), edad={edad}, conectado={conectado}")
print(f"last_sync={last_sync}")

# Conversión (como en Kotlin .toInt(), .toString())
port_str = "8000"
port_int = int(port_str)
print(f"port_str='{port_str}' -> port_int={port_int} tipo {type(port_int).__name__}")

# En Kotlin sería:
# val portStr: String = "8000"
# val portInt: Int = portStr.toInt()
# val portDouble: Double = portInt.toDouble()


### 5.3 Null Safety — El superpoder de Kotlin (y tu dolor de cabeza favorito)

En Python `None` te explota en runtime. En Kotlin el compilador te obliga a decidir:

```kotlin
// UiModels.kt:11 — health puede no existir al iniciar
val health: HealthResponse? = null // ? = puede ser null
// val health: HealthResponse = null // ❌ ERROR: no lleva ?, no puede ser null

// 3 operadores salvavidas:
val status: String? = health?.status // ?. = "si health es null, no preguntes .status, devuelve null"
val statusSeguro: String = health?.status ?: "Sin datos" // ?: Elvis = "si es null, usa esto"
val statusForzado: String = health!!.status // !! = "¡juro que no es null!" (peligroso, crashea si mientes)
```

**Cuándo usar cada uno:**
- `?.` siempre que puedas (seguro).
- `?:` para dar valor por defecto (`DashboardScreen.kt:64` `state.health?.status ?: "Sin datos"`).
- `!!` **casi nunca** — solo en tests o si acabas de hacer `if (x != null)`.

**Ejemplo real:** `DashboardScreen.kt:68` `state.health?.let { h -> Text("DB: ${h.database}") }` → "si health no es null, entra y llámalo h".


In [ ]:
# Simulando Null Safety de Kotlin en Python
health = None # HealthResponse? = null
# health = {"status": "ok", "database": "connected"}

# ?.  -> getattr con default
status = health["status"] if health else None
print(f"health?.status = {status}")

# ?: Elvis -> or
status_seguro = (health["status"] if health else None) or "Sin datos"
print(f"health?.status ?: 'Sin datos' = '{status_seguro}'")

# let -> if not None
if health is not None:
    h = health
    print(f"DB: {h['database']}")
else:
    print("(health es null, no muestro DB)")

# Ahora con health real
health = {"status": "ok", "database": "connected", "version": "1.0"}
status = health["status"] if health else None
print(f"\nCon health real: status={status}, version={health.get('version')}")
print("✅ En Kotlin: val s = health?.status ?: \"Sin datos\"")


### 5.4 Listas y Colecciones — `listOf` vs `mutableListOf`

```kotlin
// Lista INMUTABLE (no puedes añadir) — preferida
val sensores: List<String> = listOf("HC-SR04", "KY-037", "TCRT5000")
// sensores.add("LDR") // ❌ no existe add

// Lista MUTABLE (sí puedes añadir) — solo si necesitas cambiar
val eventos: MutableList<String> = mutableListOf()
eventos.add("Movimiento detectado")
eventos.add("Puerta abierta")

// Map (diccionario)
val umbrales: Map<String, Int> = mapOf("ARMED" to 300, "TRIGGERED" to 500)
val distancia: Int? = umbrales["ARMED"] // 300
```

**Cuándo usar qué:** `listOf` para datos que vienen del backend (`List<SensorEventOut>` `ApiService.kt:49`). `mutableListOf` solo para construir temporalmente (ej. acumular telemetría antes de enviar).


In [ ]:
# Listas Python vs Kotlin
# Kotlin listOf = tuple / frozenset en Python (inmutable)
sensores = ["HC-SR04", "KY-037", "TCRT5000"] # listOf en Kotlin sería inmutable
print(f"sensores: {sensores}, total {len(sensores)}")
print(f"primero: {sensores[0]}, último: {sensores[-1]}")

# mutableListOf = list normal
eventos = []
eventos.append("Movimiento detectado")
eventos.append("Puerta abierta")
print(f"eventos mutable: {eventos}")

# Map
umbrales = {"ARMED": 300, "TRIGGERED": 500}
print(f"umbral ARMED: {umbrales.get('ARMED')}")
print(f"umbral INEXISTENTE: {umbrales.get('UNKNOWN', 'no existe')} (como ?. ?: en Kotlin)")

# Operaciones típicas
filtrados = [s for s in sensores if "37" in s]
print("filtrados con '37':", filtrados)
print("  # en Kotlin: sensores.filter { it.contains('37') }")

# En Kotlin:
# val filtrados = sensores.filter { it.contains("37") }
# val nombres = sensores.map { it.lowercase() }
# val primero = sensores.firstOrNull() // no crashea si vacía


### 5.5 Condicionales — `if` y `when` (el `switch` vitaminado)

```kotlin
// if como STATEMENT (como Python)
if (!state.isConnected && !state.isLoading) {
    Text("Desconectado", color = Color.Red) // DashboardScreen.kt:55
}

// if como EXPRESIÓN (devuelve valor, no existe en Python)
val mensaje: String = if (state.isConnected) "Conectado" else "Desconectado"
val estado: String = if (health != null) health.status else "Sin datos" // DashboardScreen.kt:64

// when = switch pero poderoso (reemplaza muchos if-else)
when (val r = repo.getHealth()) { // DashboardViewModel.kt:32
    is Result.Success -> println("OK: ${r.data}")
    is Result.Error   -> println("Error: ${r.msg}")
    is Result.Loading -> println("Cargando...")
}

// when con valores
val ledColor = when (sensorType) {
    "HC-SR04" -> "Azul"
    "KY-037"  -> "Rojo"
    else       -> "Gris" // else obligatorio si no cubres todos los casos
}
```

**Tip:** `when` sin argumento sirve como `if-else` limpio: `when { x < 0 -> "neg"; x == 0 -> "cero"; else -> "pos" }`.


In [ ]:
# Condicionales Python vs Kotlin
is_connected = False
is_loading = False

if not is_connected and not is_loading:
    print("🔴 Desconectado (como DashboardScreen.kt:55)")

# if como expresión en Kotlin -> ternario en Python
mensaje = "Conectado" if is_connected else "Desconectado"
print(f"mensaje: {mensaje}")

# when -> if/elif/else o match (Python 3.10+)
class Success: pass
class Error: pass
class Loading: pass

result = Error() # simula Result.Error
if isinstance(result, Success):
    print("OK")
elif isinstance(result, Error):
    print("Error (como when is Result.Error)")
elif isinstance(result, Loading):
    print("Cargando")

# when con valores
sensor_type = "KY-037"
led_color = {"HC-SR04": "Azul", "KY-037": "Rojo"}.get(sensor_type, "Gris")
print(f"sensor {sensor_type} -> LED {led_color}")

# En Kotlin when:
# val ledColor = when(sensorType) { "HC-SR04" -> "Azul"; "KY-037" -> "Rojo"; else -> "Gris" }


### 5.6 Bucles — `for`, `while`, `repeat`

```kotlin
// for clásico
for (sensor in sensores) {
    println(sensor)
}
for (i in 0..5) println(i)        // 0,1,2,3,4,5  (incluye 5)
for (i in 0 until 5) println(i)   // 0,1,2,3,4    (excluye 5)
for (i in 5 downTo 0) println(i)  // 5,4,3,2,1,0
for (i in 0..10 step 2) println(i)// 0,2,4,6,8,10

// while
var intentos = 0
while (intentos < 3 && !isConnected) {
    repo.getHealth()
    intentos++
}

// repeat (azúcar para for)
repeat(3) { println("Reintento ${it+1}") }

// forEach con lambda
sensores.forEach { println(it) }
```

**Cuándo usar:** `for` para recorrer listas del backend, `while` para reintentos de red, `repeat` para pruebas rápidas.


In [ ]:
# Bucles Python vs Kotlin
sensores = ["HC-SR04", "KY-037", "TCRT5000"]
for s in sensores:
    print(f"sensor: {s}")

print("\n0..5 inclusive (Kotlin 0..5) -> Python range(0,6):")
for i in range(0, 6):
    print(i, end=" ")
print("\n0 until 5 (excluye 5) -> range(0,5):")
for i in range(0, 5):
    print(i, end=" ")
print("\n5 downTo 0 -> reversed:")
for i in range(5, -1, -1):
    print(i, end=" ")
print("\nstep 2 -> range(0,11,2):")
for i in range(0, 11, 2):
    print(i, end=" ")

print("\n\nwhile reintentos:")
intentos = 0
is_connected = False
while intentos < 3 and not is_connected:
    print(f" intento {intentos+1} -> getHealth()")
    intentos += 1
    if intentos == 2:
        is_connected = True # simula que al 2do conecta
print(f"conectado={is_connected} tras {intentos} intentos")

print("\nrepeat(3) en Kotlin -> for _ in range(3) en Python:")
for idx in range(3):
    print(f" Reintento {idx+1}")


### 5.7 Funciones — `fun`

```kotlin
// Básica
fun saludar(nombre: String): String {
    return "Hola, $nombre"
}

// Con valor por defecto (como Python def f(x=5))
fun getSensorEvents(limit: Int = 50, offset: Int = 0): List<SensorEventOut> { ... } // ApiService.kt:44

// De una sola línea (= en vez de { return })
fun doble(x: Int): Int = x * 2
fun esPar(x: Int): Boolean = x % 2 == 0

// suspend = puede pausarse (hace red/BD sin bloquear)
suspend fun getHealth(): Result<HealthResponse> = safeCall { api.getHealth() } // AetherRepositoryImpl.kt:20

// Extensión (añades método a una clase que no es tuya)
fun String.esUrlValida(): Boolean = this.startsWith("http://") || this.startsWith("https://")
// uso: "http://10.0.2.2:8000/".esUrlValida() -> true
```

**Interpolación:** `"Hola $nombre"` o `"DB: ${h.database}"` (`DashboardScreen.kt:69`). Equivale a `f"Hola {nombre}"` en Python.


In [ ]:
# Funciones Python vs Kotlin
def saludar(nombre: str) -> str:
    return f"Hola, {nombre}"

print(saludar("Aether"))

def get_sensor_events(limit: int = 50, offset: int = 0):
    return f"GET /sensor-events?limit={limit}&offset={offset}"

print(get_sensor_events())
print(get_sensor_events(limit=10))
print(get_sensor_events(limit=10, offset=20))

def doble(x: int) -> int:
    return x * 2
print(f"doble(7)={doble(7)}")

# suspend en Kotlin -> async en Python (conceptualmente)
import asyncio
async def get_health():
    await asyncio.sleep(0.05) # simula red
    return {"status": "ok"}

# En Kotlin: viewModelScope.launch { val r = repo.getHealth() }
# En Python: asyncio.run(get_health())
import asyncio
print(f"async get_health: {asyncio.run(get_health())}")

# Extensión Kotlin -> función helper en Python
def es_url_valida(s: str) -> bool:
    return s.startswith("http://") or s.startswith("https://")
print(f"es_url_valida('192.168.1.1')={es_url_valida('192.168.1.1')}")
print(f"es_url_valida('http://10.0.2.2:8000/')={es_url_valida('http://10.0.2.2:8000/')}")


### 5.8 Clases, `data class`, `object`, `sealed interface`

```kotlin
// class normal — como class en Python
class PreferencesManager(private val dataStore: DataStore<Preferences>) {
    suspend fun saveBaseUrl(url: String) { ... } // PreferencesManager.kt:43
}

// data class — guarda datos, te regala equals, copy, toString
data class DashboardUiState( // UiModels.kt:8
    val isLoading: Boolean = false,
    val isConnected: Boolean = false,
    val health: HealthResponse? = null,
)
val estado1 = DashboardUiState(isLoading=true)
val estado2 = estado1.copy(isLoading=false, isConnected=true) // copia cambiando solo lo que quieres

// object — singleton (una sola instancia en toda la app)
object ServiceLocator { // ServiceLocator.kt:30
    val apiService: ApiService get() = ...
}
// uso: ServiceLocator.apiService.getHealth()

// interface — contrato (qué debe hacer, no cómo)
interface AetherRepository { // AetherRepository.kt:19
    suspend fun getHealth(): Result<HealthResponse>
}

// sealed interface — enum con datos (Result solo puede ser Success o Error o Loading)
sealed interface Result<out T> { // Result.kt:10
    data class Success<T>(val data: T) : Result<T>
    data class Error(val msg: String) : Result<Nothing>
    object Loading : Result<Nothing>
}
```

**Cuándo usar:** `data class` para modelos/estado, `object` para ServiceLocator, `interface` para desacoplar (Repository), `sealed` para estados finitos (Result).


In [ ]:
# Clases Python equivalentes
from dataclasses import dataclass
from typing import Optional

@dataclass(frozen=True) # data class Kotlin
class DashboardUiState:
    is_loading: bool = False
    is_connected: bool = False
    health: Optional[dict] = None
    error: Optional[str] = None

estado1 = DashboardUiState(is_loading=True)
print(f"estado1: {estado1}")
# copy en Kotlin -> replace en Python dataclass o crear nuevo
from dataclasses import replace
estado2 = replace(estado1, is_loading=False, is_connected=True, health={"status": "ok"})
print(f"estado2 (copy): {estado2}")

# object singleton -> módulo o clase con instancia única
class _ServiceLocator:
    def __init__(self):
        self.base_url = "http://10.0.2.2:8000/"
    def get_current_base_url(self):
        return self.base_url
ServiceLocator = _ServiceLocator() # singleton
print(f"ServiceLocator URL: {ServiceLocator.get_current_base_url()}")

# sealed interface Result -> clases con herencia
from dataclasses import dataclass
class Result: pass
@dataclass
class Success(Result):
    data: object
@dataclass
class Error(Result):
    msg: str
class Loading(Result):
    pass

def handle(r: Result):
    if isinstance(r, Success):
        print(f"✅ Success: {r.data}")
    elif isinstance(r, Error):
        print(f"❌ Error: {r.msg}")
    elif isinstance(r, Loading):
        print("⏳ Loading")

handle(Success({"status": "ok"}))
handle(Error("Network error"))
handle(Loading())


### 5.9 Lambdas — Funciones anónimas (flechas)

```kotlin
// Lambda básica
val suma: (Int, Int) -> Int = { a, b -> a + b }
suma(2,3) // 5

// En Compose y colecciones se usan TODO el tiempo
sensores.filter { it.contains("HC") } // it = cada elemento
sensores.map { it.uppercase() }
Button(onClick = { vm.refreshHealth() }) { Text("Reintentar") } // DashboardScreen.kt:80

// Trailing lambda: si el último param es lambda, va fuera de ()
scope.launch { // scope.launch(block = { ... })
    val normalized = ServiceLocator.updateBaseUrl(urlInput) // DashboardScreen.kt:109
}
```

**Tip:** `it` es el nombre por defecto si la lambda tiene 1 parámetro. Si tiene 2, nombra: `{ a, b -> a + b }`.


In [ ]:
# Lambdas Python
suma = lambda a, b: a + b
print(f"suma(2,3)={suma(2,3)}")

sensores = ["HC-SR04", "KY-037", "TCRT5000", "HC-SR04"]
filtrados = list(filter(lambda s: "HC" in s, sensores))
print(f"filtrados: {filtrados}  # Kotlin: sensores.filter {{ it.contains(\"HC\") }}")

mayus = list(map(lambda s: s.upper(), sensores))
print(f"mayus: {mayus}  # Kotlin: sensores.map {{ it.uppercase() }}")

# Trailing lambda en Kotlin -> callback en Python
def launch(block):
    print("launch → ejecutando block...")
    block()

launch(lambda: print("  dentro del launch (como ServiceLocator.updateBaseUrl)"))


<a id="compose"></a>
---
## 6. 🎨 Jetpack Compose — Legos que se redibujan solos

Compose no usa XML. La UI son funciones `@Composable` que describes, Android las dibuja.

```kotlin
@Composable
fun DashboardScreen(vm: DashboardViewModel) {
    val state by vm.uiState.collectAsStateWithLifecycle() // escucha
    Column { // columna vertical
        if (!state.isConnected) Text("Desconectado", color=Color.Red)
        Text(state.health?.status ?: "Sin datos", style=MaterialTheme.typography.headlineSmall)
        if (state.isLoading) CircularProgressIndicator()
        Button(onClick={ vm.refreshHealth() }) { Text("Reintentar") }
    }
}
```

| Concepto | Qué hace | Cuándo usar |
|---|---|---|
| `@Composable` | Marca que la función dibuja UI | Siempre para pantallas/componentes |
| `Column` / `Row` | Apila vertical / horizontal | Layout básico |
| `Modifier` | Ajusta tamaño, padding, color | `Modifier.fillMaxWidth().padding(16.dp)` |
| `remember` | Guarda valor entre redibujados | `remember { mutableStateOf("") }` |
| `collectAsStateWithLifecycle()` | Convierte Flow en State para Compose | Siempre que observes ViewModel |
| `LaunchedEffect` | Efecto que se lanza UNA vez | Para animaciones o pedir datos al entrar |

**Mental model:** No pienses "imperativo" (haz A, luego B). Piensa "declarativo": *"La pantalla ES una función del estado"*. Si `state.isLoading` cambia a `true`, Compose **automáticamente** muestra el circulito.


In [ ]:
# Simulamos Compose en Python (mental model)
def Text(text, color="black"):
    print(f"  [Text color={color}] {text}")

def Button(label, on_click):
    print(f"  [Button '{label}'] (click -> {on_click.__name__})")

def CircularProgressIndicator():
    print("  [Loading spinner...]")

def DashboardScreen_compose_simulado(state):
    print("\n--- DashboardScreen recomposed ---")
    if not state["is_connected"] and not state["is_loading"]:
        Text("Desconectado", color="red")
    status = state["health"]["status"] if state["health"] else ("..." if state["is_loading"] else "Sin datos")
    Text(status)
    if state["is_loading"]:
        CircularProgressIndicator()
    if state["error"]:
        Text(state["error"], color="red")
    Button("Reintentar", on_click=lambda: print(" -> vm.refreshHealth()"))

# Simula 3 estados
for st in [
    {"is_connected": False, "is_loading": True, "health": None, "error": None},
    {"is_connected": True, "is_loading": False, "health": {"status": "ok"}, "error": None},
    {"is_connected": False, "is_loading": False, "health": None, "error": "Network error"},
]:
    DashboardScreen_compose_simulado(st)

print("\n✅ En Compose real, cada cambio de state dispara esta recomposición automáticamente.")


<a id="mvvm"></a>
---
## 7. 🔗 MVVM en AetherControl — Del código real al diagrama

```mermaid
flowchart LR
    User-->Screen[DashboardScreen.kt:35]
    Screen-- "vm.refreshHealth()" -->VM[DashboardViewModel.kt:29]
    VM-- "repo.getHealth()" -->Repo[AetherRepositoryImpl.kt:20]
    Repo-- "api.getHealth()" -->Api[ApiService.kt:28]
    Api-- "GET /health" -->Backend[(FastAPI + Postgres)]
    Backend-- "HealthResponse" -->Api
    Api-- "HealthResponse" -->Repo
    Repo-- "Result.Success" -->VM
    VM-- "_uiState.update" -->Flow[StateFlow<DashboardUiState>]
    Flow-- "collectAsState" -->Screen
```

**Mapeo 1:1 con archivos:**
- `domain/model/UiModels.kt:8` `data class DashboardUiState` — lo único que la pantalla necesita. Si añades campo, la pantalla lo ve.
- `util/Result.kt:10` `sealed interface Result` — envuelve todo lo que puede fallar.
- `data/remote/dto/HealthDto.kt` — forma exacta del JSON `{"status":"ok","database":"connected","version":"1.0"}`.
- `core/di/ServiceLocator.kt:30` — crea `OkHttpClient` + `Retrofit` + `ApiService` + `Repository` una sola vez.

**Flujo DataStore:** `DashboardScreen` escribe `urlInput` → `ServiceLocator.updateBaseUrl()` → `PreferencesManager.saveBaseUrl()` → `DataStore` → `currentBaseUrl` → nuevo `Retrofit` con la IP nueva.


In [ ]:
# Simulación MVVM completa en Python (ejecutable)
import time, random
from dataclasses import dataclass, replace
from typing import Optional

@dataclass(frozen=True)
class DashboardUiState:
    is_loading: bool = False
    is_connected: bool = False
    health: Optional[dict] = None
    error: Optional[str] = None
    last_sync: Optional[int] = None

class Result: pass
@dataclass
class Success(Result): data: object
@dataclass
class Error(Result): msg: str

# Fake ApiService (simula backend)
class FakeApiService:
    def get_health(self):
        # 30% falla como red real
        if random.random() < 0.3:
            raise IOError("Network error")
        return {"status": "ok", "database": "connected", "version": "1.0"}

# Repository con safeCall
class FakeRepository:
    def __init__(self, api): self.api = api
    def get_health(self):
        try: return Success(self.api.get_health())
        except IOError as e: return Error(str(e))
        except Exception as e: return Error(str(e))

# ViewModel simulado
class DashboardViewModel:
    def __init__(self, repo):
        self.repo = repo
        self.ui_state = DashboardUiState()
        self.refresh_health() # init { refreshHealth() }
    def refresh_health(self):
        print("VM: refreshHealth() -> loading...")
        self.ui_state = replace(self.ui_state, is_loading=True, error=None)
        result = self.repo.get_health()
        if isinstance(result, Success):
            self.ui_state = replace(self.ui_state, is_loading=False, is_connected=True, health=result.data, last_sync=int(time.time()*1000), error=None)
            print(f"VM: Success -> {result.data}")
        elif isinstance(result, Error):
            self.ui_state = replace(self.ui_state, is_loading=False, is_connected=False, error=result.msg)
            print(f"VM: Error -> {result.msg}")
        print(f"VM: nuevo estado = {self.ui_state}")

print("=== Simulación MVVM ===")
random.seed(42)
vm = DashboardViewModel(FakeRepository(FakeApiService()))
print("\n--- Usuario toca Reintentar ---")
vm.refresh_health()
vm.refresh_health()


<a id="ejercicios"></a>
---
## 8. ✏️ Ejercicios — De cero a Mini AetherControl

**Instrucciones:** Intenta resolver sin mirar la solución. Ejecuta la celda, compara tu salida con la esperada. Los ejercicios están ordenados por dificultad (1-10 ⭐).


### Ejercicio 1 — Variables `val` / `var` (⭐)
Crea `val nombre = "Aether"`, `var contador = 0`, incrementa `contador` 3 veces en un `repeat` y print "Aether: 3". En Kotlin real:
```kotlin
val nombre = "Aether"
var contador = 0
repeat(3) { contador++ }
println("$nombre: $contador")
```


In [ ]:
# Ejercicio 1 — resuélvelo aquí
nombre = "Aether" # val
contador = 0 # var
for _ in range(3):
    contador += 1
print(f"{nombre}: {contador}")
assert contador == 3, "debe ser 3"
print("✅ Correcto")


### Ejercicio 2 — Listas: filtra los sensores ruidosos (⭐⭐)
Tienes `val lecturas = listOf(50, 52, 150, 48, 51, 149, 49)` (150 son picos). Crea `val filtradas = lecturas.filter { it < 100 }` y calcula el promedio. Esperado: `50.0` aprox.


In [ ]:
lecturas = [50, 52, 150, 48, 51, 149, 49]
filtradas = [x for x in lecturas if x < 100] # Kotlin: lecturas.filter { it < 100 }
promedio = sum(filtradas) / len(filtradas)
print(f"filtradas={filtradas}, promedio={promedio:.1f}")
assert promedio == 50.0, "promedio debe ser 50.0"
print("✅ Correcto — en Kotlin: val promedio = filtradas.average()")


### Ejercicio 3 — Condicionales: semáforo de conexión (⭐⭐)
Crea función `fun estadoTexto(isConnected: Boolean, isLoading: Boolean): String` que devuelva `"Cargando..."` si loading, `"Conectado"` si connected, `"Desconectado"` si no. Usa `when` o `if` como expresión.
```kotlin
fun estadoTexto(isConnected: Boolean, isLoading: Boolean): String = when {
    isLoading -> "Cargando..."
    isConnected -> "Conectado"
    else -> "Desconectado"
}
```


In [ ]:
def estado_texto(is_connected: bool, is_loading: bool) -> str:
    # Kotlin when { isLoading -> ...; isConnected -> ...; else -> ... }
    if is_loading:
        return "Cargando..."
    elif is_connected:
        return "Conectado"
    else:
        return "Desconectado"

for caso in [(False, True), (True, False), (False, False)]:
    print(f"{caso} -> {estado_texto(*caso)}")

assert estado_texto(False, True) == "Cargando..."
assert estado_texto(True, False) == "Conectado"
assert estado_texto(False, False) == "Desconectado"
print("✅ Correcto")


### Ejercicio 4 — Bucles: EMA α=0.2 (⭐⭐⭐)
Implementa el filtro EMA del rover: `S_t = α*Y_t + (1-α)*S_{t-1}`, con `S_0 = Y_0`, `α=0.2`. Dado `raw = [50.0, 52.0, 48.0, 51.0, 49.0]`, calcula la lista `ema`. Primer valor debe ser `50.0`, segundo `50.4`.
```kotlin
fun ema(raw: List<Double>, alpha: Double = 0.2): List<Double> {
    val out = mutableListOf<Double>()
    var state: Double? = null
    for (v in raw) {
        state = if (state == null) v else alpha * v + (1 - alpha) * state
        out.add(state)
    }
    return out
}
```


In [ ]:
def ema(raw, alpha=0.2):
    out = []
    state = None
    for v in raw:
        if state is None:
            state = v
        else:
            state = alpha * v + (1 - alpha) * state
        out.append(state)
    return out

raw = [50.0, 52.0, 48.0, 51.0, 49.0]
filt = ema(raw, alpha=0.2)
print([f"{x:.2f}" for x in filt])
assert abs(filt[0] - 50.0) < 1e-9
assert abs(filt[1] - 50.4) < 1e-9
print(f"EMA: {filt}")
print("✅ Correcto — este es el mismo código que rover.ino:259 y ema_filter.py:32")


### Ejercicio 5 — Funciones + `when` + `Result` (⭐⭐⭐)
Simula `safeCall` + `Result`. Crea función `fun divide(a: Int, b: Int): Result<Int>` que devuelva `Success(a/b)` si `b!=0`, `Error("Div por cero")` si `b==0`. Luego usa `when` para imprimir.
```kotlin
sealed interface Result<out T>
data class Success<T>(val data: T): Result<T>
data class Error(val msg: String): Result<Nothing>
fun divide(a: Int, b: Int): Result<Int> = if (b==0) Error("Div por cero") else Success(a/b)
when(val r = divide(10,2)) { is Success -> println(r.data); is Error -> println(r.msg) }
```


In [ ]:
class Success:
    def __init__(self, data): self.data = data
class Error:
    def __init__(self, msg): self.msg = msg

def divide(a: int, b: int):
    if b == 0:
        return Error("Div por cero")
    return Success(a // b)

for a,b in [(10,2), (10,0), (7,3)]:
    r = divide(a,b)
    if isinstance(r, Success):
        print(f"{a}/{b} = {r.data} (Success)")
    elif isinstance(r, Error):
        print(f"{a}/{b} -> Error: {r.msg}")

assert isinstance(divide(10,2), Success)
assert isinstance(divide(10,0), Error)
print("✅ Correcto — igual que Result.kt:10 + safeCall")


### Ejercicio 6 — Integrador: Mini Dashboard (⭐⭐⭐⭐)
Combina todo: Crea `data class SensorReading(val tipo: String, val valor: Double)`. Crea `listOf` 5 lecturas, filtra solo `HC-SR04`, aplica EMA a sus valores, y usa `when` para decidir si hay obstáculo (`ema < 30` → `"¡Obstáculo!"` else `"Libre"`).


In [ ]:
from dataclasses import dataclass

@dataclass
class SensorReading:
    tipo: str
    valor: float

lecturas = [
    SensorReading("HC-SR04", 50.0),
    SensorReading("HC-SR04", 45.0),
    SensorReading("KY-037", 320.0),
    SensorReading("HC-SR04", 28.0), # obstáculo
    SensorReading("HC-SR04", 26.0),
]

# 1. filtra HC-SR04
hc = [r for r in lecturas if r.tipo == "HC-SR04"] # Kotlin: lecturas.filter { it.tipo == "HC-SR04" }
print(f"HC-SR04: {[r.valor for r in hc]}")

# 2. EMA
def ema_vals(vals, alpha=0.2):
    s=None; out=[]
    for v in vals:
        s = v if s is None else alpha*v + (1-alpha)*s
        out.append(s)
    return out

vals = [r.valor for r in hc]
filtrados = ema_vals(vals)
print(f"EMA: {[f'{x:.1f}' for x in filtrados]}")

# 3. when -> obstáculo
for v in filtrados:
    estado = "¡Obstáculo!" if v < 30 else "Libre"  # when { v < 30 -> "¡Obstáculo!"; else -> "Libre" }
    print(f"  EMA {v:.1f} -> {estado}")

print("✅ Ejercicio integrador completo — estructura igual a ViewModel + Repository")


### Ejercicio 7 — Normalización de URL (⭐⭐⭐) — Copiado de `ServiceLocator.kt:153` y `PreferencesManager.kt:63`
Escribe `fun normalizeUrl(raw: String): String` que: trim, añada `http://` si no empieza con `http://`/`https://`, y añada `/` final si falta. Prueba: `"192.168.1.50:8000"` → `"http://192.168.1.50:8000/"`.
```kotlin
fun normalizeUrl(raw: String): String {
    var u = raw.trim()
    require(u.isNotBlank()) { "URL vacía" }
    if (!u.startsWith("http://") && !u.startsWith("https://")) u = "http://$u"
    if (!u.endsWith("/")) u += "/"
    return u
}
```


In [ ]:
def normalize_url(raw: str) -> str:
    u = raw.strip()
    assert u != "", "URL vacía"
    if not (u.startswith("http://") or u.startswith("https://")):
        u = "http://" + u
    if not u.endswith("/"):
        u += "/"
    return u

tests = {
    "192.168.1.50:8000": "http://192.168.1.50:8000/",
    "http://10.0.2.2:8000/": "http://10.0.2.2:8000/",
    "10.0.2.2:8000": "http://10.0.2.2:8000/",
    " https://example.com ": "https://example.com/",
}
for inp, exp in tests.items():
    out = normalize_url(inp)
    status = "✅" if out == exp else "❌"
    print(f"{status} '{inp}' -> '{out}' (esperado '{exp}')")
    assert out == exp
print("✅ Correcto — misma lógica que ServiceLocator.kt:153")


<a id="checklist"></a>
---
## 9. ✅ Checklist y Siguientes pasos

### Checklist para el estudiante (¿puedo decir que entiendo AetherControl?)
- [ ] Puedo explicar la diferencia `val`/`var` y dar ejemplo de `DashboardScreen.kt:37` vs `DashboardScreen.kt:43`.
- [ ] Puedo leer `Result.kt:10` y explicar `Success`/`Error` con `when`.
- [ ] Puedo escribir `fun normalizeUrl` sin mirar y explicar `require`.
- [ ] Puedo dibujar el flujo MVVM de memoria: Screen → ViewModel → Repository → ApiService → Backend.
- [ ] Sé cuándo usar `listOf` (inmutable) vs `mutableListOf` (mutable).
- [ ] Sé cuándo usar `?.`, `?:` y por qué evitar `!!`.
- [ ] Entiendo que `suspend` necesita `viewModelScope.launch` y que `StateFlow` se observa con `collectAsStateWithLifecycle()`.
- [ ] Puedo implementar `ema()` con `for` y `var state: Double? = null`.

### Siguientes pasos (para el repo)
1. **Agregar pantalla Rover:** Crear `ui/screens/RoverScreen.kt` + `RoverViewModel` siguiendo el mismo patrón `UiModels.kt:8` → `DashboardViewModel.kt:18`.
2. **Persistir más prefs:** Añadir `Keys.ROVER_SPEED` en `PreferencesManager.kt:23`.
3. **Migrar a Room:** Definir `@Entity` en `AppDatabase.kt` para cache offline de `SensorEventOut`.
4. **Tests:** Extender `DashboardViewModelTest.kt` para probar `Error` y `Loading` con `turbine`.
5. **Compose avanzado:** Usar `rememberSaveable` para `urlInput` y `LaunchedEffect` para animar el banner "Desconectado".

### Referencias del proyecto
- `docs/prd.md` — Visión, KPIs, restricciones FOSS
- `docs/requirements.md` — RF-1.1, RNF-3.1, HU BDD
- `docs/architecture.md` — Diagrama de capas
- `docs/sprints.md` — Qué toca en cada sprint (no adelantar Sprint 3)
- `app/build.gradle.kts` — Dependencias (Retrofit, Compose BOM, DataStore, Turbine)
- `stats/notebooks/EMA_Estadistica.ipynb` — Notebook hermano: EMA estadístico (Python) — ver cómo `ema()` se usa igual en firmware y en app.

> **Tip final:** Si algo no compila, lee el error en rojo de Android Studio — 90% de las veces es un `?` faltante o un `val` que intentas cambiar. ¡El compilador es tu amigo, no tu enemigo!
